In [1]:
import pandas as pd
import tabulate

## Load and Import Admissions Data

In [ ]:
admissions = pd.DataFrame(pd.read_csv("admissions.csv.gz"))

print(f"Shape of admissions dataframe: {admissions.shape}")

print(f"Number of unique patients: {admissions["subject_id"].nunique()}")
print(f"Number of total patient admissions: {admissions["hadm_id"].nunique()}")

Shape of admissions dataframe: (275, 16)
Number of unique patients: 100
Number of total patient admissions: 275


In [3]:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaN,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,NaN,NaN,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaN,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,NaN,UNKNOWN,NaN,NaN,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaN,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,NaN,NaN,0


## Load and Import Diagnoses Data

In [4]:
diagnoses = pd.DataFrame(pd.read_csv("diagnoses_icd.csv.gz"))
print(f"Shape of admissions dataframe: {diagnoses.shape}")
print(f"Number of unique admissions in diagnoses table: {diagnoses["hadm_id"].nunique()}")
print(f"Number of total diagnoses in diagnoses table: {diagnoses["hadm_id"].size}")

Shape of admissions dataframe: (4506, 5)
Number of unique admissions in diagnoses table: 275
Number of total diagnoses in diagnoses table: 4506


In [5]:
print(diagnoses.head())

   subject_id   hadm_id  seq_num icd_code  icd_version
0    10035185  22580999        3     4139            9
1    10035185  22580999       10     V707            9
2    10035185  22580999        1    41401            9
3    10035185  22580999        9     3899            9
4    10035185  22580999       11    V8532            9


In [6]:
# Checking to see split of data between icd_version 9 and icd_version 10
print(diagnoses[diagnoses["icd_version"]==9].shape)
print(diagnoses[diagnoses["icd_version"]==10].shape)

(2193, 5)
(2313, 5)


## Merge Admissions and Diagnoses

In [8]:
#Combining admissions table and diagnoses table along 'hadm_id'
patient_data = pd.merge(admissions, diagnoses, on=["subject_id", "hadm_id"])
print(patient_data.shape)

(4506, 19)


In [9]:
patient_data.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,seq_num,icd_code,icd_version
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,15,42731,9
1,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,5,51881,9
2,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,12,75169,9
3,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,13,4254,9
4,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,21,2749,9


In [10]:
# Sanity check to ensure data merged properly
print(f"Does row count of patient_data table match the row count of diagnoses table? {diagnoses.shape[0]==patient_data.shape[0]} ")

Does row count of patient_data table match the row count of diagnoses table? True 


## Add Readable Diagnosis Names

In [11]:
# Pulling data for conversion table between icd_code and long_title
diag_names = pd.DataFrame(pd.read_csv("d_icd_diagnoses.csv.gz"))
print(f"Shape of diagnosis conversion table: {diag_names.shape}")
print('--------------------')
diag_names.head()


Shape of diagnosis conversion table: (109775, 3)
--------------------


,icd_code,icd_version,long_title
0,0090,9,"Infectious colitis, enteritis, and gastroenter..."
1,01160,9,"Tuberculous pneumonia [any form], unspecified"
2,01186,9,"Other specified pulmonary tuberculosis, tuberc..."
3,01200,9,"Tuberculous pleurisy, unspecified"
4,01236,9,"Tuberculous laryngitis, tubercle bacilli not f..."


In [12]:
# Adding diagnoses' long titles to our patient_data table, ensuring to match icd code and version
new_patient_data = pd.merge(patient_data, diag_names, on=['icd_code', 'icd_version'])
new_patient_data.head()



,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,seq_num,icd_code,icd_version,long_title
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,15,42731,9,Atrial fibrillation
1,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,5,51881,9,Acute respiratory failure
2,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,12,75169,9,"Other anomalies of gallbladder, bile ducts, an..."
3,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,13,4254,9,Other primary cardiomyopathies
4,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,21,2749,9,"Gout, unspecified"


In [13]:
# Sanity check to ensure no dropped data
print(f"Do the lengths of old patient data table and new patient data table match? {new_patient_data.shape[0]==patient_data.shape[0]}")

Do the lengths of old patient data table and new patient data table match? True


## Analysis: Diagnosis Frequency

In [14]:
#Grouping and sorting to view the top 10 highest diagnoses

diag_occur = new_patient_data.groupby('long_title')
diag_occur_counts = diag_occur.size().sort_values(ascending=False)
highest_diagnoses = diag_occur_counts.reset_index(name='Number_of_Diagnoses').head(10)
highest_diagnoses.rename(columns = {'long_title':'Diagnosis_Name'}, inplace=True)


In [15]:
print(highest_diagnoses)

                                Diagnosis_Name  Number_of_Diagnoses
0           Unspecified essential hypertension                   68
1                  Hyperlipidemia, unspecified                   57
2            Acute kidney failure, unspecified                   56
3         Other and unspecified hyperlipidemia                   55
4                  Hypothyroidism, unspecified                   47
5                         Obesity, unspecified                   43
6                          Anemia, unspecified                   41
7           Long term (current) use of insulin                   37
8  Urinary tract infection, site not specified                   36
9      Personal history of nicotine dependence                   35


## Analysis: Length of Stay Based on Diagnosis

In [16]:
# Converting admit time and discharge time from strings into time datatypes
new_patient_data["admittime"] = pd.to_datetime(new_patient_data["admittime"])
new_patient_data["dischtime"] = pd.to_datetime(new_patient_data["dischtime"])



In [17]:
# Calculating duration of stay for every patient.
new_patient_data["Length_of_Stay"] = new_patient_data["dischtime"] - new_patient_data["admittime"]

In [18]:
new_patient_data.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,marital_status,race,edregtime,edouttime,hospital_expire_flag,seq_num,icd_code,icd_version,long_title,Length_of_Stay
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,15,42731,9,Atrial fibrillation,8 days 23:24:00
1,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,5,51881,9,Acute respiratory failure,8 days 23:24:00
2,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,12,75169,9,"Other anomalies of gallbladder, bile ducts, an...",8 days 23:24:00
3,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,13,4254,9,Other primary cardiomyopathies,8 days 23:24:00
4,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,...,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0,21,2749,9,"Gout, unspecified",8 days 23:24:00


In [19]:
stay_grouped = new_patient_data.groupby("long_title")

sizes = stay_grouped.size()
stay_indexed = sizes[sizes>5].index

filt_mean = stay_grouped["Length_of_Stay"].mean().loc[stay_indexed]
filt_median = stay_grouped["Length_of_Stay"].median().loc[stay_indexed]


In [20]:
print("Top ten highest average lengths of time admitted per diagnosis:")
print("----------------------------------------------------------------------------")
mean_times = filt_mean.sort_values(ascending=False).head(10).reset_index(name='Average_Time')
mean_times.rename(columns={'long_title':'Diagnosis_Name'}, inplace=True)
mean_times['Average_Time'] = mean_times['Average_Time'].dt.round('min')
print(mean_times)


Top ten highest average lengths of time admitted per diagnosis:
----------------------------------------------------------------------------
                                      Diagnosis_Name     Average_Time
0           Unspecified protein-calorie malnutrition 21 days 10:48:00
1  Other medical procedures as the cause of abnor... 18 days 02:12:00
2                                 Acute pancreatitis 17 days 13:06:00
3    Unspecified severe protein-calorie malnutrition 17 days 05:19:00
4                    Ventilator associated pneumonia 16 days 20:18:00
5                             Dysphagia, unspecified 16 days 19:29:00
6         Acute kidney failure with tubular necrosis 16 days 14:34:00
7               Hyperosmolality and/or hypernatremia 16 days 03:16:00
8     Accidents occurring in residential institution 14 days 19:20:00
9  Patient room in hospital as the place of occur... 14 days 08:02:00


In [21]:
print("Top ten highest median lengths of time admitted per diagnosis:")
print("---------------------------------------------------------------------")
median_times = filt_median.sort_values(ascending=False).head(10).reset_index(name='Average_Time')
median_times.rename(columns={'long_title':'Diagnosis_Name'}, inplace=True)
median_times['Average_Time'] = median_times['Average_Time'].dt.round('min')
print(median_times)

Top ten highest median lengths of time admitted per diagnosis:
---------------------------------------------------------------------
                                      Diagnosis_Name     Average_Time
0           Unspecified protein-calorie malnutrition 23 days 15:13:00
1         Acute kidney failure with tubular necrosis 17 days 05:06:00
2    Unspecified severe protein-calorie malnutrition 16 days 14:52:00
3                                 Acute pancreatitis 14 days 20:27:00
4                    Ventilator associated pneumonia 14 days 13:50:00
5  Type 2 diabetes mellitus with hypoglycemia wit... 13 days 18:26:00
6                         Physical restraints status 13 days 14:12:00
7                    Coagulation defect, unspecified 13 days 09:26:00
8           Type 2 diabetes mellitus with foot ulcer 13 days 02:14:00
9  Acute kidney failure with lesion of tubular ne... 13 days 00:47:00


## Export Results

In [22]:
results = f'''
# Results

## Diagnosis Frequency (Top 10)
{highest_diagnoses.to_markdown()}

## Mean Length of Stay by Diagnosis (Top 10)
{mean_times.to_markdown()}

## Median Length of Stay by Diagnosis (Top 10)
{median_times.to_markdown()}

'''

with open("results.md", "w") as file:
    file.write(results)